# 07 — Arc Analysis

Detect and quantify the rise-and-fall of abstract language across literary history.

Uses two regression approaches:
- **Quadratic**: `score ~ year + year²` — tests for inverted-U, estimates peak year
- **Piecewise linear**: two slopes with optimal breakpoint — measures abstracting/concretizing rates

In [ ]:
import sys
sys.path.insert(0, '..')
from abstraction.analysis import *
from abstraction.plotting import plot_arc, plot_arc_by_genre

## Load all scored corpora

Single load used throughout the notebook.

In [ ]:
combined = load_all_scored()
print(f'{len(combined)} texts, {combined.year.min():.0f}-{combined.year.max():.0f}, {combined.corpus_name.nunique()} corpora')
combined.genre_harmonized.value_counts()

## Canon Fiction: full arc

In [ ]:
# Filter canon fiction from combined
df = combined[combined.corpus_name == 'canon_fiction'].copy()
print(f'{len(df)} texts, {df.year.min()}-{df.year.max()}')
df[['id', 'year', 'major_genre', 'Abs-Conc.Median.median']].head()

In [ ]:
# Fit the arc (all years)
result = fit_arc(df)
print(summarize_arc(result))

In [ ]:
# Restrict to 1500+ (avoid ancient outliers pulling the peak)
result_1500 = fit_arc(df, min_year=1500)
print(summarize_arc(result_1500))

## Across norm columns

Does the arc hold regardless of which norm we use?

In [ ]:
score_cols = get_score_columns(df)
rows = []
for col in score_cols:
    r = fit_arc(df[df.year >= 1500], score_col=col)
    r['score_col'] = col
    rows.append(r)

norms_df = pd.DataFrame(rows)
# Show: which norms show the expected arc (β₂ > 0 = concreteness U = abstractness inverted-U)
norms_df['has_arc'] = (norms_df['quad_beta2'] > 0) & (norms_df['quad_beta2_p'] < 0.05)
summary = norms_df[['score_col', 'quad_beta2', 'quad_beta2_p', 'quad_peak_year', 'quad_r2', 'has_arc']]
print(f"Norms showing expected arc: {summary.has_arc.sum()} / {len(summary)}")
summary.sort_values('quad_r2', ascending=False)

## By genre

Does the arc appear within individual genres?

In [ ]:
genre_rows = []
for genre, gdf in df[df.year >= 1600].groupby('major_genre'):
    if len(gdf) < 30:
        continue
    r = fit_arc(gdf, score_col='Abs-Conc.Median.median')
    r['genre'] = genre
    genre_rows.append(r)

genre_df = pd.DataFrame(genre_rows)
genre_df['has_arc'] = (genre_df['quad_beta2'] > 0) & (genre_df['quad_beta2_p'] < 0.05)
for _, row in genre_df.iterrows():
    row_d = row.to_dict()
    row_d['corpus'] = row_d.get('genre', '?')
    print(summarize_arc(row_d))
    print()

## All scored corpora

Run arc fitting across every corpus in `data/scores/v7/`.

In [ ]:
# All corpora, per-corpus arc fitting (reuses combined)
all_results = fit_arc_all_corpora(combined_df=combined, min_year=1600)
all_results['has_arc'] = (all_results['quad_beta2'] > 0) & (all_results['quad_beta2_p'] < 0.05)
all_results[['corpus', 'n_texts', 'year_min', 'year_max', 'quad_peak_year', 'quad_beta2_p', 'quad_r2', 'has_arc', 'pw_break_year', 'pw_slope_before', 'pw_slope_after']].sort_values('quad_r2', ascending=False)

## Cross-corpus arc by genre (with corpus fixed effects)

Pool all scored corpora, harmonize genres, and fit one arc per genre with corpus dummy variables absorbing baseline differences.

In [ ]:
genre_results = fit_arc_all_by_genre(combined_df=combined, corpus_fixed_effects=True)
for _, row in genre_results.iterrows():
    print(summarize_arc(row.to_dict()))
    print()

## Corpus-adjusted arc plots

Remove corpus-level baseline differences and plot the shared time trend. Adjusted points are shifted to a common baseline; the black line is the fitted trend (quadratic or piecewise).

In [ ]:
# Single genre: Fiction (quadratic)
fiction = combined[combined.genre_harmonized == 'Fiction']
adj = adjust_scores(fiction, model='quadratic')
plot_arc(adj, title='Fiction (quadratic, corpus-adjusted)')

In [ ]:
# Single genre: Fiction (piecewise)
adj_pw = adjust_scores(fiction, model='piecewise')
plot_arc(adj_pw, title='Fiction (piecewise, corpus-adjusted)')

In [ ]:
genres = {'Periodical', 'Poetry', 'Fiction', 'Drama'}
combined2 = combined[combined.genre_harmonized.isin(genres)]

In [ ]:
# All genres, faceted (quadratic)
plot_arc_by_genre(combined2, model='quadratic')

In [ ]:
# All genres, faceted (piecewise)
plot_arc_by_genre(combined2, model='piecewise')